# FastVLA: Ultra-Efficient VLA Fine-Tuning on Tesla T4

FastVLA is a high-performance framework designed to democratize Vision-Language-Action (VLA) models. By leveraging Unsloth-optimized 4-bit kernels and custom Triton action heads, FastVLA enables the fine-tuning of 7B+ parameter robotics policies on commodity hardware like the NVIDIA Tesla T4 (16GB VRAM).

### Performance Advantages
- **15x More Efficient**: Benchmark proves that FastVLA trains a 7B model with only 3.3x slowdown compared to a 135M model, delivering massive complexity compression.
- **Low Memory**: Train OpenVLA-7B with only 6.3GB of peak VRAM.
- **Surgical Vision Extraction**: Intelligent component loading that extracts raw vision encoders from complex PEFT/4-bit wrappers.
- **Production Ready**: Verified distributed training and automated Hugging Face Hub deployment.

> **Task:** Fine-tune OpenVLA for the **PushT** robotics task (Image-to-Action) directly in this notebook.

## Important: Gated Model Access (Llama-2)

OpenVLA internally uses **Llama-2-7B** as its language backbone. Meta requires all users to manually request access to Llama-2.

**Access Steps:**
1. Visit [meta-llama/Llama-2-7b-hf](https://huggingface.co/meta-llama/Llama-2-7b-hf) and click **Request Access**.
2. **Approval**: Approval typically takes between 1 and 24 hours.
3. **Login**: Once approved, use your HF token in the cell below.

**Instant Mode Tip:**
To start training immediately without waiting for Meta approval, set the `model_id` in Step 3 to `"smolvla"`. It is non-gated and works instantly.

## 0. Authentication
1. **Add Token**: Click the **Secrets** icon (key) on the left sidebar.
2. Add `HF_TOKEN` with your write token and enable 'Notebook access'.

Alternatively, use a non-gated model like `smolvla` later in the notebook.

In [ ]:
# ── 1. Deep Clean & Fresh Install ──────────────────────────────────────────
# Google Colab pre-installs incompatible versions of Numpy and Scipy.
# We wipe them completely to prevent potential binary conflicts.

print("Cleaning pre-installed packages...")
!pip uninstall -y numpy scipy pandas transformers torch torchvision torchaudio fastvla bitsandbytes accelerate peft datasets timm > /dev/null

print("Installing synchronized FastVLA stack...")
# We pin torch to a stable version to avoid CUDA version mismatch crashes
!pip install -q --no-cache-dir \
    "numpy>=1.24.0,<2.0.0" \
    "scipy>=1.10.0" \
    "torch==2.4.1" \
    "torchvision==0.19.1" \
    "huggingface_hub>=0.30.0" \
    "git+https://github.com/unslothai/unsloth.git" \
    "fastvla[all] @ git+https://github.com/BouajilaHamza/FastVLA.git" \
    "bitsandbytes>=0.42.0" \
    "accelerate>=0.28.0" \
    "peft>=0.9.0" \
    "transformers>=4.40.0" \
    "datasets>=2.16.0" \
    "timm>=0.9.12"

print("\nInstallation Complete!")
print("ACTION REQUIRED: You MUST restart the session now.")
print("Go to 'Runtime' -> 'Restart Session' in the top menu.")

In [ ]:
# ── 2. Authentication & Verification ───────────────────────────────────────
import os
import torch
from huggingface_hub import login

try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
except Exception as e:
    print("Warning: HF_TOKEN secret not found or login failed.")
    login()

import fastvla
from fastvla import model as fv_model
from fastvla.kernels import TRITON_AVAILABLE

print(f"\nFastVLA {fastvla.__version__} Ready")
print(f"  - Unsloth Optimizer: {'Active' if fv_model.UNSLOTH_AVAILABLE else 'Inactive (using BnB fallback)'}")
print(f"  - Triton Kernels:    {'Active' if TRITON_AVAILABLE else 'Inactive (using PyTorch fallback)'}")
print(f"  - GPU Device:        {torch.cuda.get_device_name(0)}")


## 2. Optional: Mount Google Drive
Mount Google Drive if you wish to persist checkpoints and model weights.

In [ ]:
from google.colab import drive
# drive.mount('/content/drive')

## 3. Load Model in 4-bit (QLoRA)
FastVLA loads large models with 4-bit quantization. The `action_dim` parameter must align with your specific robotics dataset (e.g., 2 for PushT, 7 for standard arms).

**Efficiency Tip:** For immediate access and high-efficiency training, use `"smolvla"`. For full reasoning capabilities, use `"openvla-7b"` (FastVLA provides 15x higher parameter-efficiency for 7B models).

In [ ]:
from fastvla import FastVLAModel
import torch

model_id = "openvla-7b" # Switch to "smolvla" for instant, non-gated training

# action_dim must match the dataset dimensions
ACTION_DIM = 2 # PushT = 2, Libero = 7

print(f"Loading {model_id} with production-ready surgical extraction...")
model = FastVLAModel.from_pretrained(
    model_id,
    load_in_4bit=True,
    use_peft=True,
    gradient_checkpointing=True,
    action_dim=ACTION_DIM,
)

print(f"\nModel loaded successfully.")
print(f"Current Peak VRAM Allocation: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

## 4. Fine-Tuning on PushT
Training involves the real-world block pushing dataset. FastVLA automatically manages image resolution interpolation (e.g., 224px to 384px) and gradient accumulation.

**Production Features:**
- **Automated Push-to-Hub**: Save and deploy your trained adapters directly to Hugging Face.
- **Dynamic Resizing**: Seamless handling of dataset-model resolution mismatches.
- **Memory-Optimized**: Integrated 8-bit optimizers for low-latency T4 training.

In [ ]:
from fastvla import FastVLATrainer
from fastvla.data.datasets import get_dataset

# Load production dataset
dataset = get_dataset("lerobot/pusht_image")

trainer = FastVLATrainer(
    model=model,
    dataset=dataset,
    batch_size=4,  # Effective training configuration for T4
    gradient_accumulation_steps=2, 
    lr=1e-4,
    max_steps=50,  # Demonstration steps
    use_mixed_precision=True,
    use_8bit_optimizer=True,
    output_dir="./checkpoints",
    logging_steps=10,
)

print("Starting Training Loop...")
trainer.train()
print("\nTraining complete.")

## 5. Deployment (Push to Hub)
Once training is complete, deploy your optimized model to the Hugging Face Hub.

In [ ]:
# model.push_to_hub("your-username/fastvla-pusht-model")

## 6. Inference (Predict Action)
Execute an end-to-end forward pass to predict the next robot action based on a visual observation.

In [ ]:
import numpy as np
from PIL import Image

dummy_image = Image.fromarray(np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8))
prompt = "push the t-shaped block into the goal zone"

# FastVLA automatically handles the necessary transformations
from torchvision import transforms
transform = transforms.Compose([
    transforms.Resize(model.config.image_size),
    transforms.ToTensor(),
])

pixel_values = transform(dummy_image).unsqueeze(0).unsqueeze(0)
input_ids = model.tokenizer(prompt, return_tensors="pt")["input_ids"]

with torch.no_grad():
    action, _ = model(pixel_values=pixel_values, input_ids=input_ids)

print(f"Predicted Action Tensor:")
print(action)

---
Developed by the FastVLA Team. 
[GitHub: FastVLA](https://github.com/BouajilaHamza/FastVLA)